# Gradient Boosting and Model Comparison

🎯 **Where this lesson takes you.** In Lesson 3 you turned a linear baseline into a **Random Forest** — many trees grown *independently* and averaged (bagging) — and learned to evaluate it honestly with cross-validation, tuning, and feature importance. This lesson introduces the other great ensemble family, **boosting**, where trees are grown *sequentially*, each one correcting the last one's mistakes. You'll train a `GradientBoostingClassifier`, put it head-to-head with the Random Forest and the baseline, then do the thing that actually matters in practice: **save the winning model, build a reusable prediction script, and make a defensible business recommendation** that weighs the cost of a missed bankruptcy against a false alarm.

## What you'll be able to do

By the end of this notebook you will be able to:

-   Explain **boosting** and how it differs from **bagging** (Random
    Forests).
-   Train a `GradientBoostingClassifier` for bankruptcy prediction.
-   Compare **baseline**, **Random Forest**, and **Gradient Boosting**
    models.
-   Evaluate models with imbalanced metrics: precision, recall, F1,
    ROC–AUC (+ a confusion matrix and classification report).
-   Build a complete prediction pipeline: `wrangle()` → train → predict
    → save model → load model.
-   Make a practical recommendation: *Which model would you trust and
    why?* (consider business consequences)

> 💡 **Tip — decoding the features.** The model inputs are named `feat_1 … feat_64`, which tells you nothing about what they measure. If you want the human-readable meaning of each `feat_*` column, open `data-dictionary.ipynb`. You don't need it to train, but it's essential when you interpret results and explain them to a stakeholder.

# 1. Conceptual Foundation

The single new idea this lesson is **boosting**. The cleanest way to understand it is by contrast with the bagging you already know from Random Forests — same goal (combine many weak trees into one strong model), opposite strategy.

## Bagging vs. boosting (big picture)

Both bagging and boosting combine many “weak” models into a stronger
one, but they do it in different ways.

### Bagging (Random Forest mindset)

-   Train many trees **independently**, each on a bootstrap sample.
-   Average / vote across trees.
-   Goal: reduce **variance** (stability, generalization).

### Boosting (Gradient Boosting mindset)

-   Train models **sequentially**.
-   Each new model focuses on the mistakes of the previous ones.
-   Goal: reduce **bias** (fit complex patterns progressively).

**Practical intuition:**

-   Random Forest: “many independent opinions”
-   Gradient Boosting: “a team improving iteratively from feedback”

Side by side, the contrast is sharp — and it explains *why* the two families fail and succeed in different ways:

| | Bagging (Random Forest) | Boosting (Gradient Boosting) |
|---|---|---|
| **How trees are built** | independently, in parallel | sequentially, each fixes the last |
| **What each tree sees** | a bootstrap sample of rows | the *residual errors* so far |
| **Primarily reduces** | variance | bias |
| **Failure mode** | can stay slightly underpowered | can overfit if pushed too hard |
| **Parallelizable?** | yes (trees are independent) | no (each stage needs the previous) |

➡️ **Verdict.** Bagging makes an unstable model *stable* by averaging; boosting makes a weak model *strong* by relentless error-correction. Because boosting keeps chasing residuals, it can squeeze out more accuracy — but that same hunger is exactly what makes it prone to overfitting, which is why the *knobs* below matter so much.

## Why boosting can help for financial risk problems

Boosting often performs well when:

-   signals are weak in any single feature, but strong in combinations
-   relationships are nonlinear and involve interactions
-   you can accept a slightly more complex model to gain performance

However, boosting can also overfit if the model is too complex or
trained too aggressively (too many estimators, large learning rate).

💡 **Why this fits bankruptcy data.** No single financial ratio screams "this firm will fail." The signal lives in *combinations* — leverage rising while liquidity falls while margins compress. Boosting's sequential error-correction is well suited to teasing out exactly these faint, interacting patterns, which is why it's a perennial favorite on tabular risk problems.

## Key knobs in Gradient Boosting

-   `n_estimators`: number of boosting stages (more can help, but can
    overfit)
-   `learning_rate`: step size for each stage (smaller is safer but
    needs more estimators)
-   `max_depth`: complexity of each weak learner (depth 1–3 is common)
-   `subsample`: using \< 1.0 can reduce overfitting (stochastic
    boosting)

Rule of thumb: **small `learning_rate` + more estimators** tends to be
safer.

> 🔧 **The two knobs that interact most.** `learning_rate` and `n_estimators` pull against each other: a small learning rate takes timid steps, so you need *more* stages to converge — but those small, numerous steps generalize better than a few aggressive ones. `max_depth` controls how complex each weak learner is (shallow stumps are typical for boosting, unlike the deep trees in a forest), and `subsample < 1.0` injects randomness for extra regularization. We'll grid-search a small, readable combination of these below.

➡️ **From theory to practice.** That's the whole new concept. The rest of the notebook reuses the Lesson-3 workflow — load Poland, split, oversample, define a metric helper — then trains three models, compares them, interprets the errors, and ships the winner.

------------------------------------------------------------------------

🎥 **Walkthrough video.** Before the hands-on work, watch the short walkthrough for this lesson. Run the cell below to load it.

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1170263287", h="3298dbabb7", width=700, height=450) 

## A quick peek at a `.json.gz` file (why gzip/json matters)

In earlier notebooks you used `wrangle()` from `data.py`. That function
hides the raw file details. Here is a small reminder of what the raw
files look like.

🔍 **A quick callback to Lesson 1.** `wrangle()` does a lot of invisible work: open the gzip, parse the JSON, normalize nested records into a flat table. This optional peek pops the hood for one moment so the file stops being a black box — you'll see the top-level keys that `wrangle()` knows how to unpack.

**Code 5.4.1.1**:

In [ ]:
# Inspect the *structure* of a compressed JSON file (optional exploration).
import gzip
import json
from pathlib import Path

sample_path = Path("data/poland-bankruptcy-data-2009.json.gz")

with gzip.open(sample_path, "rt", encoding="utf-8") as f:
    payload = json.load(f)

# The top-level keys show how records are stored inside the file.
list(payload.keys())

📊 **Reading the keys.** The top-level keys reveal how the records are organized inside the compressed file — typically metadata alongside the array of per-company observations. This is the structure `wrangle()` flattens into the tidy `feat_*` table you'll model on.

Key points:

-   [`gzip.open`](https://docs.python.org/3/library/gzip.html#gzip.open)
-   [`json.load`](https://docs.python.org/3/library/json.html#json.load)
-   [`pathlib.Path`](https://docs.python.org/3/library/pathlib.html)

# Applied Exercises

## 2. Setup

We import everything the lesson needs up front: `pickle` for saving the model, `ipywidgets` for the interactive threshold slider, the imbalanced-learn oversampler, and the scikit-learn estimators, metrics, and model-selection tools. As in Lesson 3, `wrangle()` comes from `data.py`.

**Code 5.4.2.1**:

In [ ]:
# Imports and display settings used throughout the notebook.
import pickle
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from ipywidgets import interact
from matplotlib import pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline

from data import wrangle

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 120)

## 3. Load Poland (full dataset with target)

### Problem

Before we can train any model, we need a clean table of financial
features plus the target label (`bankrupt`) for each firm.

### Approach

Load the Poland dataset using `Path` and `wrangle()`, then confirm the
target column exists.

Key points:

-   [`pathlib.Path`](https://docs.python.org/3/library/pathlib.html)
-   [`pandas.DataFrame.shape`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shape.html)
-   [`assert`
    statement](https://docs.python.org/3/reference/simple_stmts.html#the-assert-statement)

**Code 5.4.3.1**:

In [ ]:
# Point to the Poland dataset file and load it with wrangle().
poland_path = Path("data/poland-bankruptcy-data-2009.json.gz")
poland_df = wrangle(poland_path)

# Quick sanity-check: (rows, columns)
poland_df.shape

📊 **Reading the shape.** The `(rows, columns)` tuple confirms you loaded the labeled file: one row per Polish firm, 64 `feat_*` ratios plus the `bankrupt` target. This is the table all three models will share.

### Checkpoint

✅ This `assert` fails loudly if `bankrupt` is missing — which would mean you grabbed the *unlabeled* file. Passing silently means the supervised target is present and you're clear to continue.

**Code 5.4.3.2**:

In [ ]:
# Checkpoint: ensure the supervised-learning target is present.
assert "bankrupt" in poland_df.columns, (
    "Expected 'bankrupt'. Make sure you loaded the full Poland dataset."
)

## 4. Create `X` (features) and `y` (target)

### Problem

Scikit-learn expects features and target to be separated so we can train
models and evaluate predictions.

### Approach

In this section, you will turn the cleaned Poland DataFrame into the two
objects that every scikit-learn classifier expects:

1.  **Identify the feature columns (`feature_cols`)** Scan the DataFrame
    columns and keep only the financial indicators whose names start
    with `"feat_"`. This ensures you include *only* numeric model inputs
    and exclude identifiers or the target label by mistake.

2.  **Build the feature matrix (`X`)** Use `feature_cols` to select a
    **2D** pandas DataFrame containing only the financial features. This
    is what you will pass as `X` to `fit()` and `predict()`.

3.  **Build the target vector (`y`)** Extract the `bankrupt` column as a
    **1D** pandas Series and cast it to integers so it becomes a clean
    binary label vector (`0`/`1`) for classification.

4.  **Sanity-check shapes and assumptions (checkpoint)** Print shapes
    once to confirm that `X` and `y` align, then run the provided
    asserts to verify you have the expected number of features for
    Poland and that the target contains only valid binary values.

These variables will be reused in the train/test split and every model
that follows.

Key points:

-   [`str.startswith`](https://docs.python.org/3/library/stdtypes.html#str.startswith)
-   [`pandas.Series.astype`](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html)
-   scikit-learn convention:
    [`fit(X, y)`](https://scikit-learn.org/stable/glossary.html#term-fit)

**Code 5.4.4.1**:

In [ ]:
# 1) Select the financial feature columns.
#    We keep only columns whose names start with "feat_" so we don't
#    accidentally include the target or any non-feature columns.
feature_cols = [c for c in poland_df.columns if c.startswith("feat_")]

# 2) Build X as a DataFrame with just the features.
#    X must be 2D (rows = firms, columns = financial indicators).
X = poland_df[feature_cols]

# 3) Build y as an integer Series (0/1).
#    y must be 1D and represent the label we want to predict.
#    Casting to int ensures labels are clean and consistent for metrics/models.
y = poland_df["bankrupt"].astype(int)

# Quick sanity check: X should have the same number of rows as y.
print(X.shape, y.shape)

📊 **Reading the shapes.** `X` should report 64 columns and the same row count as `y`. Holding `feature_cols` as an explicit list matters later — the `predictor.py` script in §11 rebuilds `X` from exactly this `feat_*` rule so training and deployment stay in sync.

### Checkpoint

✅ Two guardrails: the feature matrix must have exactly 64 columns, and the target must be strictly binary `{0, 1}`. A failure here means something upstream is wrong and every model below would train on a broken shape.

**Code 5.4.4.2**:

In [ ]:
# Checkpoint: validate the dataset structure before training models.
# Poland is expected to have exactly 64 financial features (feat_1..feat_64).
assert X.shape[1] == 64, "Poland should have feat_1..feat_64"

# Checkpoint: ensure the target is strictly binary after casting.
assert set(y.unique()).issubset({0, 1})

## 5. Train/test split (stratified)

### Problem

We need a test set that reflects the same rare-bankruptcy rate as the
full dataset. Otherwise, the evaluation metrics can be misleading.

### Approach

In this section, you will create one reproducible train/test split that
you will reuse for *every* model later in the notebook.

1.  **Split features and labels into train and test sets** Call
    `train_test_split` on `X` and `y`, choosing a `test_size` (here 25%)
    so you keep enough data to train while still reserving a meaningful
    holdout set for evaluation.

2.  **Preserve the rare-bankruptcy rate with stratification** Pass
    `stratify=y` so the proportion of bankrupt vs. non-bankrupt firms is
    approximately the same in both `y_train` and `y_test`. This keeps
    evaluation fair and prevents accidentally creating a test set with
    too few (or too many) bankruptcies.

3.  **Make the split reproducible** Set `random_state` so you (and
    students/graders) get the same split each run, which makes results
    comparable and debugging easier.

4.  **Verify the split with a quick rate comparison + checkpoint**
    Compute the mean of `y_train` and `y_test` (since labels are 0/1,
    the mean is the bankruptcy rate) and confirm with an assert that the
    two rates are very close.

Key points:

-   [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
-   `stratify` argument: same link above

🔄 **One split, reused everywhere.** This is the discipline that makes the three-model comparison fair: every model below trains and is scored on the *same* `X_train`/`X_test`. If each model got its own random split, differences in the metrics could be luck of the draw rather than real skill.

**Code 5.4.5.1**:

In [ ]:
# Split once and reuse these arrays for all models below.
# This ensures every model is trained and evaluated on the same data split.
X_train, X_test, y_train, y_test = train_test_split(
    X,             # feature matrix (2D)
    y,             # target labels (1D)
    test_size=0.25,  # keep 25% as a held-out test set
    random_state=42, # reproducible split across runs
    stratify=y,      # preserve class proportions in train and test
)

# Compare bankruptcy rates to confirm stratification worked.
# Because y is 0/1, the mean equals the fraction of bankrupt firms.
y_train.mean(), y_test.mean()

📊 **Reading the two rates.** Because the labels are 0/1, each `.mean()` *is* the bankruptcy rate. The train and test rates should be nearly identical — that's stratification working. Both will be small, a numeric reminder of how rare the positive class is.

### Checkpoint

✅ The payoff of `stratify=y`: train and test bankruptcy rates must differ by less than one percentage point, or the test set wouldn't be a fair stand-in for reality.

**Code 5.4.5.2**:

In [ ]:
# Checkpoint: train/test bankruptcy rates should be very close.
# If this fails, the split may not be stratified or something is off in y.
assert abs(y_train.mean() - y_test.mean()) < 0.01

## 6. Oversample the training data

### Problem

With heavy class imbalance, many models can become conservative and
predict too few bankruptcies, hurting recall.

### Approach

In this section, you will rebalance the training data so the model sees
enough bankruptcy examples during learning—without contaminating the
test set.

1.  **Create the oversampler** Instantiate `RandomOverSampler` with a
    fixed `random_state` so the resampling is reproducible.

2.  **Fit and resample *only* the training split** Call
    `fit_resample(X_train, y_train)` to generate `X_train_over` and
    `y_train_over`, where the minority class (bankruptcies) is
    duplicated until both classes have the same number of samples.
    Keeping this step restricted to the training set avoids leaking
    information into evaluation.

3.  **Confirm the new class balance** Inspect
    `y_train_over.value_counts()` to verify that the two classes are now
    (approximately) equal in count. These oversampled arrays will be
    used to fit models in later sections, while `X_test`/`y_test` remain
    unchanged for a realistic test.

Key points:

-   [`RandomOverSampler`](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.RandomOverSampler.html)
-   [`fit_resample`](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.RandomOverSampler.html#imblearn.over_sampling.RandomOverSampler.fit_resample)

> ⚠️ **Same leakage rule as Lesson 3 — training set only.** Oversample *after* the split, never before. If duplicated bankruptcies leak into the test set, the model effectively sees its own answers and the metrics become fiction. We resample `X_train`/`y_train` and leave `X_test`/`y_test` untouched. Note the split here: the two Gradient Boosting models train on the **oversampled** data, while the Random Forest uses `class_weight="balanced"` on the **original** data — two routes to the same goal, exactly as you compared in Lesson 3.

**Code 5.4.6.1**:

In [ ]:
# Oversample the minority class in the training set only.
# This increases the number of bankrupt examples by duplicating them
# until both classes have the same count.
ros = RandomOverSampler(
    random_state=42  # reproducible resampling
)

# Fit the oversampler on the training data and return the resampled arrays.
# IMPORTANT: do NOT resample X_test/y_test, or evaluation will be unrealistic.
X_train_over, y_train_over = ros.fit_resample(
    X_train,  # original training features
    y_train,  # original training labels
)

# After resampling, the class counts should be balanced (same number of 0s and 1s).
y_train_over.value_counts()

📊 **Reading the resampled counts.** After `fit_resample`, the two class counts should be equal — the minority bankruptcies have been duplicated up to the majority count. The training set is now larger, and crucially the test set is unchanged.

### Checkpoint

✅ Confirms the oversampler did its job: the two class counts must now be identical. If they weren't, the "balanced" training set wouldn't actually be balanced.

**Code 5.4.6.2**:

In [ ]:
# Checkpoint: oversampling should make both classes equally frequent.
counts = y_train_over.value_counts()
assert counts.iloc[0] == counts.iloc[1]

## 7. Define a reusable metric helper

### Problem

We will evaluate multiple models. Repeating metric code increases the
chance of mistakes and makes comparisons harder.

### Approach

In this section, you will create a single helper function that
standardizes how we score every classifier in the notebook.

1.  **Define a function with a consistent input contract** Implement
    `compute_metrics(y_true, y_pred, y_proba)` where:

    -   `y_true` is the ground-truth label vector (`0/1`),
    -   `y_pred` is the model’s predicted class labels (`0/1`),
    -   `y_proba` is the model’s predicted probability for the positive
        class (bankruptcy = `1`). Keeping these inputs consistent makes
        it easy to plug in any model later.

2.  **Compute the same core metrics for every model** Inside the
    function, calculate:

    -   **precision** (how many predicted bankruptcies are truly
        bankrupt),
    -   **recall** (how many true bankruptcies we successfully catch),
    -   **F1** (balance between precision and recall),
    -   **ROC–AUC** (how well probabilities rank bankruptcies vs.
        non-bankruptcies).

3.  **Return results in a structured format** Return a dictionary
    mapping metric names to floats. This makes it easy to:

    -   print results cleanly,
    -   build tables later,
    -   compare multiple models side-by-side without rewriting scoring
        code.

4.  **Make the helper robust for rare-event predictions** Use
    `zero_division=0` for precision/recall/F1 so the function behaves
    predictably even when a model produces no positive predictions
    (which can happen with heavy imbalance).

We will reuse it for every model.

Key points:

-   [`precision_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
-   [`recall_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)
-   [`f1_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
-   [`roc_auc_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)

🧱 **One helper, three fair comparisons.** Writing the scoring logic once and reusing it for every model is the antidote to subtle bugs — a stray `zero_division` here, a probability-vs-label mix-up there. The contract `(y_true, y_pred, y_proba)` is deliberate: hard labels feed precision/recall/F1, while **probabilities** feed ROC–AUC. Mixing those up is one of the most common evaluation mistakes.

**Code Task 5.4.7.1**:

In [ ]:
def compute_metrics(
    y_true: pd.Series,
    y_pred: pd.Series,
    y_proba: pd.Series,
) -> dict[str, float]:
    """Compute core metrics for imbalanced binary classification.

    Parameters
    ----------
    y_true
        True labels (0/1).
    y_pred
        Predicted labels (0/1).
    y_proba
        Predicted probabilities for class 1.

    Returns
    -------
    dict[str, float]
        Precision, recall, F1, ROC–AUC.
    """
    # Precision: of the predicted bankruptcies, how many were correct?
    # zero_division=0 avoids errors when a model predicts zero positives.
    precision = float(precision_score(y_true, y_pred, zero_division=0))

    # Recall: of the true bankruptcies, how many did we detect?
    recall = float(recall_score(y_true, y_pred, zero_division=0))

    # F1: harmonic mean of precision and recall (useful under imbalance).
    f1 = float(f1_score(y_true, y_pred, zero_division=0))

    # ROC–AUC: evaluates how well predicted probabilities rank positives above negatives.
    # This uses y_proba (probability of class 1), not hard class labels.
    roc_auc = float(roc_auc_score(y_true, y_proba))

    # Return a consistent dictionary so all models can be compared uniformly.
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    }

## 8. Train and evaluate three models

### Problem

We want to compare a bagging model (Random Forest) against a boosting
model (Gradient Boosting) using a consistent evaluation setup.

### Approach

In this section, you will train and evaluate three models using the
*same* test set and the *same* metric helper, so results are directly
comparable.

1.  **Set up consistent preprocessing with a pipeline** For every model,
    use `make_pipeline(SimpleImputer(strategy="median"), model)` so
    missing values are handled the same way each time. Median imputation
    is a reasonable default for numeric financial features and avoids
    dropping rows.

2.  **Train a baseline Gradient Boosting model (boosting baseline)** Fit
    a `GradientBoostingClassifier` with simple/default-ish settings on
    the **oversampled training set** (`X_train_over`, `y_train_over`).
    Then predict:

    -   `y_pred` using `predict(X_test)` for class labels,
    -   `y_proba` using `predict_proba(X_test)[:, 1]` for positive-class
        probabilities. Finally, compute and store metrics with
        `compute_metrics(...)`.

3.  **Train a Random Forest reference model (bagging reference)** Fit a
    `RandomForestClassifier` on the **original training set**
    (`X_train`, `y_train`) while using `class_weight="balanced"` to
    compensate for class imbalance without oversampling. Use the same
    prediction and scoring steps as above to get `rf_metrics`.

4.  **Tune Gradient Boosting with a small grid search** Create a
    Gradient Boosting pipeline (`gb_pipe`) and define a focused
    `param_grid` over a few impactful hyperparameters (number of trees,
    learning rate, depth, subsampling). Run `GridSearchCV` with:

    -   `scoring="roc_auc"` to optimize probability ranking under
        imbalance,
    -   `cv=3` to keep it lightweight but meaningful. Fit the grid
        search on the **oversampled training set** to keep the tuning
        objective consistent with the baseline boosting model.

5.  **Evaluate the best tuned model on the untouched test set** Extract
    `grid.best_estimator_`, generate predictions on `X_test`, compute
    metrics with `compute_metrics`, and store them as `gb_metrics`. At
    the end, you will have three comparable metric dictionaries:
    `gb0_metrics`, `rf_metrics`, and `gb_metrics`.

Key points:

-   [`make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
-   [`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html)
-   [`GradientBoostingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html)
-   [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
-   [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

🧪 **The experiment, set up cleanly.** Three models, one test set, one scorer. The boosting models learn from the oversampled data; the forest leans on `class_weight="balanced"`. Holding everything else constant is what lets the comparison table in §9 mean something — any difference you see is the *model*, not the plumbing.

**Code Task 5.4.8.1**:

In [ ]:
# Baseline Gradient Boosting (train on oversampled data).
# We use a pipeline so missing values are handled consistently via median imputation.
gb_baseline = make_pipeline(
    SimpleImputer(strategy='median'),  # fill missing numeric values
    GradientBoostingClassifier(random_state=42),  # boosting model (baseline)
)

# Fit on the oversampled training set so the model sees enough bankrupt examples.
gb_baseline.fit(X_train_over, y_train_over)

# Predict hard class labels (0/1) on the untouched test set.
y_pred_gb0 = gb_baseline.predict(X_test)

# Predict probabilities for the positive class (bankrupt = 1).
# We take [:, 1] because column 1 corresponds to class 1 probabilities.
y_proba_gb0 = gb_baseline.predict_proba(X_test)[:, 1]

# Compute the core metrics using the helper for consistent evaluation.
gb0_metrics = compute_metrics(y_test, y_pred_gb0, y_proba_gb0)
gb0_metrics

📊 **Reading the baseline boosting metrics.** This is Gradient Boosting with default-ish settings — your boosting starting point. Note its recall and ROC–AUC; tuning (step 4) has to beat *these* numbers to justify the extra search cost.

**Code 5.4.8.2**:

In [ ]:
# Random Forest reference model (bagging).
# Here we train on the original (non-oversampled) training split and use
# class_weight="balanced" to compensate for imbalance during learning.
rf = make_pipeline(
    SimpleImputer(strategy="median"),  # same preprocessing as other models
    RandomForestClassifier(
        n_estimators=300,       # number of trees in the forest
        random_state=42,        # reproducibility
        n_jobs=-1,              # use all CPU cores
        class_weight="balanced" # weight classes inversely proportional to frequency
    ),
)

# Fit on the original training set (no oversampling here).
rf.fit(X_train, y_train)

# Predict hard labels on the test set.
y_pred_rf = rf.predict(X_test)

# Predict probabilities for class 1 (bankrupt).
y_proba_rf = pd.Series(rf.predict_proba(X_test)[:, 1])

# Score with the same helper to enable fair comparisons.
rf_metrics = compute_metrics(y_test, y_pred_rf, y_proba_rf)
rf_metrics

📊 **Reading the Random Forest reference.** This is the bagging champion from Lesson 3, retrained on the same split. It's the model boosting must outperform to earn its place. Compare its recall and ROC–AUC against the baseline boosting above.

**Code Task 5.4.8.3**:

In [ ]:
# Tuned Gradient Boosting using a small, readable grid.
# We keep the same preprocessing step to ensure tuning compares apples-to-apples.
gb_pipe = make_pipeline(
    SimpleImputer(strategy='median'),
    GradientBoostingClassifier(random_state=42),
)

# Grid over a few key hyperparameters:
# Fast grid: hold three parameters fixed and compare two tree depths.
# This reduces 16 configurations to 2 (6 CV fits instead of 48).


param_grid = {
    "gradientboostingclassifier__n_estimators": [100],
    "gradientboostingclassifier__learning_rate": [0.1],
    "gradientboostingclassifier__max_depth": [2, 3],
    "gradientboostingclassifier__subsample": [0.8],
}

# Use ROC–AUC for selection because it evaluates probability ranking
# and is commonly used for imbalanced binary classification.
grid = GridSearchCV(
    gb_pipe,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=3,     # small CV for speed while still validating generalization
    n_jobs=-1 # parallelize across CPU cores
)

# Fit the search on the oversampled training data (same as baseline GB training).
grid.fit(X_train_over, y_train_over)

# Inspect the best hyperparameters and best cross-validated ROC–AUC.
grid.best_params_, grid.best_score_

📊 **Reading the search result.** `best_params_` is the winning mix of `n_estimators`, `learning_rate`, `max_depth`, and `subsample`; `best_score_` is its mean cross-validated ROC–AUC on the training folds. Watch whether the search favored a *small* learning rate with *more* estimators — the "safer" combination the rule of thumb predicted.

**Code 5.4.8.4**:

In [ ]:
# Evaluate the best tuned model on the test set.
# best_estimator_ is the full pipeline (imputer + tuned gradient boosting).
gb_best = grid.best_estimator_

# Predict test labels and probabilities using the tuned pipeline.
y_pred_gb = gb_best.predict(X_test)
y_proba_gb = pd.Series(gb_best.predict_proba(X_test)[:, 1])

# Compute test metrics for the tuned model.
gb_metrics = compute_metrics(y_test, y_pred_gb, y_proba_gb)
gb_metrics

📊 **Reading the tuned model on test.** Now the tuned boosting model meets the **untouched** test set. The honest question: did tuning actually move recall/ROC–AUC above the baseline boosting and the forest, or did it just chase noise in the CV folds? The comparison table next makes the answer unambiguous.

## 9. Compare models in a table

### Problem

It is hard to compare models when metrics are scattered across many
outputs.

### Approach

In this section, you will consolidate the metric dictionaries from each
model into a single, readable comparison table.

1.  **Gather each model’s metrics in a consistent structure** You
    already computed three dictionaries (`rf_metrics`, `gb0_metrics`,
    `gb_metrics`) using the same `compute_metrics` helper. Here, you
    will combine them into a list of row dictionaries where:

    -   `"model"` provides the row label,
    -   the metric dictionary (precision, recall, F1, ROC–AUC) becomes
        the row’s columns.

2.  **Create a DataFrame from the list of rows** Pass the list of
    dictionaries into `pd.DataFrame(...)`. Pandas will align keys across
    rows to form columns, giving you one column per metric.

3.  **Set the model name as the index for clean display** Use
    `.set_index("model")` so model names become row labels rather than a
    regular column. This makes the table easier to read and reference.

4.  **Validate the table has all required metrics (checkpoint)** Run the
    assert to ensure the expected metric columns exist. This protects
    you against typos in metric keys or missing results from earlier
    steps.

Key points:

-   [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
-   [`DataFrame.set_index`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.set_index.html)

**Code Task 5.4.9.1**:

In [ ]:
# Combine metrics into a single comparison table.
# Each list item becomes one row in the final DataFrame.
# We store a readable model name under "model" and then unpack the metric dict.
comparison = pd.DataFrame(
    [
        {"model": "Random Forest", **rf_metrics},         # bagging reference
        {"model": "GradBoost (baseline)", **gb0_metrics},  # boosting baseline
        {"model": "GradBoost (tuned)", **gb_metrics},      # tuned boosting model
    ]
)

# Use the model name as the row index so the table reads naturally.
comparison = comparison.set_index("model")

# Display the comparison table (rows = models, columns = metrics).
comparison

📊 **The headline comparison.** This is the table the whole notebook builds toward: three models, four metrics, one glance. Read it by *column*, not by row — there is rarely one model that wins everything. A model with higher **recall** catches more bankruptcies (fewer costly misses); a model with higher **precision** raises fewer false alarms. Which column matters more is a business decision, not a statistical one — we settle it in §10 and the wrap-up.

### Checkpoint

✅ Confirms the comparison table carries all four expected metric columns — a guard against a typo in a metric key or a missing result from an earlier step.

**Code 5.4.9.2**:

In [ ]:
# Checkpoint: confirm that all expected metric columns are present.
# This helps catch missing keys or earlier metric computation issues.
assert {"precision", "recall", "f1", "roc_auc"}.issubset(comparison.columns)

## 10. Interpret errors (confusion matrix and report)

### Problem

Aggregate metrics can hide important error patterns. For bankruptcy
prediction, the business cost of a **false negative** (missed
bankruptcy) is often much higher than a **false positive**.

### Approach

In this section, you will go beyond aggregate scores and inspect *which
kinds of mistakes* the tuned model is making on the test set.

1.  **Visualize the confusion matrix (tuned Gradient Boosting)** Use the
    tuned model’s test predictions (`y_pred_gb`) to build a confusion
    matrix. This breaks outcomes into:

    -   **True Negatives**: correctly predicted non-bankrupt,
    -   **False Positives**: predicted bankrupt but actually not (false
        alarm),
    -   **False Negatives**: missed bankruptcies (often the most
        costly),
    -   **True Positives**: correctly predicted bankrupt. The plot makes
        it easy to spot whether the model is failing mainly by missing
        bankruptcies (FN) or raising too many false alarms (FP).

2.  **Print a classification report to see per-class performance** Use
    `classification_report` to view precision/recall/F1 *for each
    class*, rather than a single overall number. Pay special attention
    to the metrics for class `1` (bankruptcy), because that class is
    rare and typically the one you care about detecting.

3.  **Optionally explore different probability thresholds** The default
    decision rule is threshold `0.5` (predict class 1 when
    `P(bankrupt) >= 0.5`). In imbalanced problems, you often adjust this
    cutoff:

    -   **Lower threshold** → catch more bankruptcies (higher recall)
        but more false positives,
    -   **Higher threshold** → fewer false positives but more missed
        bankruptcies. The provided helper function recomputes
        predictions, prints key metrics, and shows how the confusion
        matrix changes as you move the threshold slider.

Key points:

-   [`ConfusionMatrixDisplay.from_predictions`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html)
-   [`classification_report`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)
-   [`confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
-   [`ipywidgets.interact`](https://ipywidgets.readthedocs.io/en/stable/examples/Using%20Interact.html)

**Code 5.4.10.1**:

In [ ]:
# Confusion matrix + report for the tuned Gradient Boosting model.
# The confusion matrix shows counts of TN / FP / FN / TP on the test set.
ConfusionMatrixDisplay.from_predictions(
    y_test,        # true labels
    y_pred_gb,     # predicted labels from the tuned model at the default threshold
    values_format="d",  # show integer counts
)

# Add a descriptive title so the plot is self-explanatory in the notebook.
plt.title("Tuned Gradient Boosting — Confusion Matrix (Test)")
plt.show()

# The classification report gives precision/recall/F1 for each class (0 and 1).
# digits=3 improves readability; zero_division=0 avoids warnings if a class
# has no predicted positives.
print(classification_report(y_test, y_pred_gb, digits=3, zero_division=0))

> ⚠️ **Read the bottom-left cell first.** In this confusion matrix the **false negatives** — firms that went bankrupt but the model labeled "safe" — are the expensive errors. A bank that trusts a false negative keeps lending to a company that then collapses. Count that cell, then check the classification report's **recall for class 1**: that single number is the share of real bankruptcies the model actually caught. For risk screening, it usually matters more than overall accuracy or even precision.

**Code 5.4.10.2**:

In [ ]:
# Optional: explore thresholds to see the precision/recall trade-off.
def threshold_demo(threshold: float = 0.5) -> None:
    """Display metrics and a confusion matrix for a chosen threshold."""
    # Convert probabilities into class predictions using the chosen cutoff.
    # If proba >= threshold => predict bankruptcy (1), else non-bankrupt (0).
    y_pred_thr = (y_proba_gb >= threshold).astype(int)

    # Compute the confusion matrix counts for this threshold.
    cm = confusion_matrix(
        y_test,     # true labels
        y_pred_thr, # thresholded predictions
    )

    # Print key metrics so you can see how the threshold shifts the trade-off.
    print(
        "precision:",
        precision_score(y_test, y_pred_thr, zero_division=0),
        "recall:",
        recall_score(y_test, y_pred_thr, zero_division=0),
        "f1:",
        f1_score(y_test, y_pred_thr, zero_division=0),
    )

    # Plot the confusion matrix for the chosen threshold.
    ConfusionMatrixDisplay(cm).plot(values_format="d")
    plt.title(f"Threshold = {threshold:.2f}")
    plt.show()

🔧 **What this helper lets you do.** `threshold_demo` rebuilds predictions at *any* cutoff instead of the default `0.5`, then reprints precision/recall/F1 and redraws the matrix. It's the machinery behind the slider in the next cell — change the threshold and watch the false-negative and false-positive cells trade off in real time.

**Code 5.4.10.3**:

In [ ]:
# Try a few thresholds and observe how the confusion matrix changes.
# Use a slider to interactively choose the cutoff and rerun threshold_demo.
interact(
    threshold_demo,
    threshold=widgets.FloatSlider(
        value=0.5,  # default scikit-learn threshold
        min=0.05,   # lower cutoff to prioritize recall
        max=0.95,   # higher cutoff to prioritize precision
        step=0.05,  # coarse steps for quick exploration
    ),
);

> 🚦 **The threshold is a business lever, not a fixed constant.** Drag the slider **down** and the model predicts bankruptcy more readily: recall climbs (you miss fewer failing firms) but false alarms rise. Drag it **up** and you get the opposite. There is no statistically "correct" cutoff — the right one depends on what your organization pays for each kind of mistake. That is exactly the trade-off we quantify in the recommendation below.

## 11. Build a full prediction pipeline (save/load)

### Problem

In practice, you usually train a model once and then reuse it later for
batch prediction or deployment.

### Approach

1.  Save the tuned model to disk with `pickle`.
2.  Load it back and confirm predictions match the original model.

Key points:

-   [`pickle.dump`](https://docs.python.org/3/library/pickle.html#pickle.dump)
-   [`pickle.load`](https://docs.python.org/3/library/pickle.html#pickle.load)
-   [`Path.mkdir`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.mkdir)
-   [`numpy.allclose`](https://numpy.org/doc/stable/reference/generated/numpy.allclose.html)

📦 **From notebook to artifact.** A trained model is only useful if you can use it again without re-running the whole notebook. `pickle` serializes the *entire fitted pipeline* — imputer plus tuned booster — into a single `.pkl` file. The save/load round-trip below, verified by a checkpoint, is the smallest honest version of "deploying" a model.

**Code Task 5.4.11.1**:

In [ ]:
# Create a folder to store trained models.
model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

# Save the tuned Gradient Boosting model to .
model_path = model_dir / "poland_gradient_boosting.pkl"

with model_path.open('wb') as f:
    pickle.dump(gb_best, f)

model_path

📊 **What just happened.** The fitted pipeline is now a file on disk (`models/poland_gradient_boosting.pkl`). The displayed path is your saved artifact — everything the model needs to score new firms, frozen in one object.

**Code 5.4.11.2**:

In [ ]:
# Load the model back from disk.
with model_path.open("rb") as f:
    loaded_model = pickle.load(f)

📦 **Reloading proves portability.** Reading the pickle back gives you a `loaded_model` that's a fresh, independent copy of the tuned pipeline — the same object you'd get in a separate script or server process with no access to this notebook's memory.

**Code 5.4.11.3**:

In [ ]:
# Predict again on X_test to confirm the loaded model works.
loaded_proba = loaded_model.predict_proba(X_test)[:, 1]
loaded_pred = (loaded_proba >= 0.5).astype(int)

loaded_metrics = compute_metrics(
    y_test,
    pd.Series(loaded_pred),
    pd.Series(loaded_proba),
)
loaded_metrics

📊 **Reading the reloaded metrics.** Scoring the reloaded model on `X_test` should reproduce the tuned model's numbers exactly. If they match, the round-trip is lossless — the saved artifact behaves identically to the in-memory model.

### Checkpoint

✅ The decisive test: `np.allclose` confirms the reloaded model's probabilities are numerically identical to the original's. If this passes, the `.pkl` is a faithful copy and safe to deploy.

**Code 5.4.11.4**:

In [ ]:
# Checkpoint: loaded model should reproduce the same probabilities.
assert np.allclose(loaded_proba, y_proba_gb.values)

### Optional: Write a minimal script you could use for batch prediction.

A simple way to make your work reusable is to extract the “inference”
steps into a tiny script that can be run anytime, without re-opening the
notebook. The idea is to load a saved model from disk, apply the exact
same preprocessing via `wrangle()`, build `X` from the `feat_*` columns,
and then output bankruptcy probabilities for every row in the input
file.

Write a standalone `predictor.py` that exposes a single function,
`predict_bankruptcy_prob(...)`, so you can either import it from another
Python program or run it directly as a script (the `__main__` block).
This is the same pattern you would use for batch prediction jobs: keep
training separate, keep prediction lightweight, and make it easy to
point the script at a new dataset file whenever you need fresh risk
scores.

🔧 **Why a separate script, not just the notebook.** Notebooks are for exploration; production prediction wants a small, importable function. `predictor.py` reuses the *same* `wrangle()` and the *same* `feat_*` selection rule from §4, so the data a model sees in deployment is built identically to the data it trained on — the single most common source of silent production bugs is a train/serve preprocessing mismatch, and this pattern closes that gap.

**Code 5.4.11.5**:

In [ ]:
predictor_code = '''
from __future__ import annotations

import pickle
from pathlib import Path

import pandas as pd

from data import wrangle

def predict_bankruptcy_prob(
    model_path: str | Path,
    data_path: str | Path,
) -> pd.Series:
    """Predict bankruptcy probabilities for a dataset file.

    Parameters
    ----------
    model_path
        Path to a trained scikit-learn pipeline saved with pickle.
    data_path
        Path to a dataset file compatible with ``wrangle``.

    Returns
    -------
    pd.Series
        Predicted probabilities for class 1 (bankrupt), indexed by row id.
    """
    model_path = Path(model_path)
    data_path = Path(data_path)

    with model_path.open("rb") as f:
        model = pickle.load(f)

    df = wrangle(data_path)
    feat_cols = [c for c in df.columns if c.startswith("feat_")]
    X = df[feat_cols]

    proba = model.predict_proba(X)[:, 1]
    return pd.Series(proba, index=df.index)

if __name__ == "__main__":
    model_path = Path("models/poland_gradient_boosting.pkl")
    data_path = Path("data/poland-bankruptcy-data-2009.json.gz")

    probs = predict_bankruptcy_prob(model_path, data_path)
    print(probs.head())
'''

Path("predictor.py").write_text(predictor_code, encoding="utf-8")
Path("predictor.py")

📊 **What you've built.** `predictor.py` now exists on disk as a self-contained inference module: point it at a model file and a dataset, get back a Series of bankruptcy probabilities. This is the deployable end-to-end artifact the lesson promised — `wrangle()` → load model → predict, with no notebook required.

## 12. Apply the same pipeline to Taiwan (Optional/Ungraded)

In this section, you can apply what you learned in the previous sections
to analyze the Taiwan dataset. This section is optional and ungraded,
but it is highly recommended to reinforce what you have learned so far.

💡 **Why repeat on Taiwan.** The Taiwan data has a different feature set and a different imbalance ratio. Re-running the full load → split → oversample → train → compare → interpret workflow there confirms you've learned the *process*, not just the Poland numbers — exactly the transferable skill this project is training.

**Code 5.4.12.1**:

In [ ]:
# your code here

------------------------------------------------------------------------

# Wrap-up

In this notebook, you learned the core idea behind **boosting** and how
it differs from **bagging**:

-   **Random Forest (bagging)** builds many trees independently and
    aggregates them to improve stability and reduce variance.
-   **Gradient Boosting (boosting)** builds trees sequentially, where
    each new tree focuses on correcting the errors of the previous ones.

You trained and evaluated a **Gradient Boosting** model and compared it
against a **Random Forest** model and a simple baseline. You also
practiced a clean, reproducible workflow for applied machine learning:
loading data with `wrangle`, splitting into train/test sets, handling
imbalance (with oversampling on the training set only), evaluating with
multiple metrics, and summarizing results in a comparison table.

Most importantly, you practiced the real skill behind applied modeling:
**making decisions based on evidence**. Different models can look
“better” under different metrics, so the right choice depends on your
objective—especially the cost of false negatives vs. false positives in
a business setting.

> ⚠️ **The recommendation hinges on asymmetric costs — and the asymmetry is large.** The two mistakes a bankruptcy model can make are not equally expensive, so you cannot pick a model (or a threshold) on F1 or accuracy alone. Make the asymmetry concrete with an illustrative example:
>
> - **False negative** (model says "safe," firm goes bankrupt): the lender keeps its exposure and can lose the **entire** outstanding loan — *potentially the full principal, say on the order of hundreds of thousands of euros per firm.*
> - **False positive** (model says "risky," firm is actually fine): the cost is a **manual review** by an analyst, or a declined-but-recoverable opportunity — *on the order of hours of staff time, perhaps a few hundred euros.*
>
> *(These figures are illustrative, to show the shape of the trade-off — not outputs of this notebook.)* When one error can cost a thousand times more than the other, the rational policy is clear: **favor recall**. Choose the model — and lower the decision threshold (§10) — to catch as many true bankruptcies as you can afford, accepting more false alarms as the cheap price of avoiding catastrophic misses.

🎯 **How to actually choose.** Bring it together in one decision rule: read the §9 table for the model with the strongest **recall and ROC–AUC** (ROC–AUC tells you it ranks risky firms well *at any* threshold), confirm in §10 that its false-negative cell is acceptable, then tune the threshold to the cost ratio above. Report the choice the way a stakeholder needs to hear it: *"This model catches X% of failing firms; each one it misses risks the full loan, while each false alarm costs only a review — so we accept the extra reviews."* That sentence, backed by the evidence you generated here, is the deliverable.

➡️ **Where this goes next.** You now have the full applied-ML loop — data in, compared models out, a saved artifact, and a cost-aware recommendation. That is the complete arc of Project 5. The same skeleton (wrangle → split → handle imbalance → compare → interpret → deploy → recommend) is the template you'll carry into every future modeling project.